# multiply-back composite — cx16: register multiply_back0 / multiply_back1 in the lookup

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `backward-func-lookup`, `multiply-back`, `arg-position-back-functions`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "multiply-back"
DD_ATOM_IDS = ["backward-func-lookup", "multiply-back", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: BackwardFuncLookup", "Backprop: multiply_back", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing multiply_back (with unbroadcast) into the BackwardFuncLookup

`multiply` is mathematically symmetric (`x*y = y*x`), but the manual autograd still registers TWO back fns — `multiply_back0` for arg-0 and `multiply_back1` for arg-1 — because the dispatcher doesn't know any op is symmetric. It just looks up `(fn, argnum)`.

Both bodies follow the SAME pattern: local derivative * grad_out, then `unbroadcast(...)` to collapse any broadcast axes back to the parent's shape. Then both register into the `BackwardFuncLookup` under `t.multiply` at argnum 0 and 1.

This composite has you wire the full registration end-to-end: write the back fns, register them, and then dispatch by `(t.multiply, argnum)` to compute grads.

### Composite Exercise — register multiply_back0 / multiply_back1 in the lookup

**Atoms exercised together**: `backward-func-lookup`, `multiply-back`, `arg-position-back-functions`

Implement:

**1. `BackwardFuncLookup`** with `add_back_func` and `get_back_func` (raises `KeyError` on miss).

**2. `multiply_back0(grad_out, out, x, y)`** — returns `unbroadcast(grad_out * y, x)`, shaped like `x`.

**3. `multiply_back1(grad_out, out, x, y)`** — returns `unbroadcast(grad_out * x, y)`, shaped like `y`.

**4. `cx16_build_lookup()`** — returns a populated `BackwardFuncLookup` with both back fns registered under `t.multiply` at argnum 0 and 1 (use `add_back_func`, not direct dict access).

The `unbroadcast(grad, original)` helper is provided. The test exercises lookup + dispatch + value + broadcast collapse.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def unbroadcast(grad, original):
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

class BackwardFuncLookup:
    def __init__(self):
        raise NotImplementedError
    def add_back_func(self, forward_fn, arg_position, back_fn):
        raise NotImplementedError
    def get_back_func(self, forward_fn, arg_position):
        raise NotImplementedError

def multiply_back0(grad_out, out, x, y):
    raise NotImplementedError

def multiply_back1(grad_out, out, x, y):
    raise NotImplementedError

def cx16_build_lookup():
    """Return a BackwardFuncLookup with both multiply_back fns registered under t.multiply."""
    raise NotImplementedError

def _test_cx16():
    BF = cx16_build_lookup()
    assert isinstance(BF, BackwardFuncLookup)

    # (a) both argnums resolve and the registry is keyed by t.multiply
    f0 = BF.get_back_func(t.multiply, 0)
    f1 = BF.get_back_func(t.multiply, 1)
    assert f0 is multiply_back0
    assert f1 is multiply_back1
    assert f0 is not f1

    # (b) same-shape multiply: grads are simply y and x scaled by grad_out
    x = t.tensor([2.0, 3.0, 4.0]); y = t.tensor([5.0, 6.0, 7.0])
    out = x * y
    g0 = f0(t.ones(3), out, x, y)
    g1 = f1(t.ones(3), out, x, y)
    assert g0.shape == x.shape and g1.shape == y.shape
    assert t.allclose(g0, y) and t.allclose(g1, x)

    # (c) BROADCAST: x shape (1,4) * y shape (3,4) -> out (3,4); grad must collapse back
    x_b = t.tensor([[1.0, 2.0, 3.0, 4.0]])
    y_b = t.tensor([[5.0, 6.0, 7.0, 8.0],
                     [9.0, 10.0, 11.0, 12.0],
                     [13.0, 14.0, 15.0, 16.0]])
    out_b = x_b * y_b
    grad_b = t.ones(3, 4)
    g0_b = f0(grad_b, out_b, x_b, y_b)
    g1_b = f1(grad_b, out_b, x_b, y_b)
    assert g0_b.shape == x_b.shape, (
        f'multiply_back0 forgot unbroadcast: shape {g0_b.shape} != {x_b.shape}'
    )
    assert g1_b.shape == y_b.shape
    expected_g0 = (grad_b * y_b).sum(dim=0, keepdim=True)
    assert t.allclose(g0_b, expected_g0)
    assert t.allclose(g1_b, grad_b * x_b.expand_as(y_b))

    # (d) cross-check the broadcast case vs torch.autograd
    xr = x_b.clone().requires_grad_(True)
    yr = y_b.clone().requires_grad_(True)
    (xr * yr).sum().backward()
    g0_v = f0(t.ones(3, 4), xr.detach() * yr.detach(), xr.detach(), yr.detach())
    g1_v = f1(t.ones(3, 4), xr.detach() * yr.detach(), xr.detach(), yr.detach())
    assert t.allclose(g0_v, xr.grad), f'disagree x: {g0_v} vs {xr.grad}'
    assert t.allclose(g1_v, yr.grad), f'disagree y: {g1_v} vs {yr.grad}'

    # (e) dispatcher-style: argnum comes from a parents dict like the real reverse pass would build
    parents = {0: x, 1: y}
    grads = {idx: BF.get_back_func(t.multiply, idx)(t.ones(3), out, x, y) for idx in parents}
    assert t.allclose(grads[0], y) and t.allclose(grads[1], x)
    _dd_passed.add('cx16')

_test_cx16()

<details><summary>Show solution — cx16</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}
    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn
    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn for ({forward_fn!r}, argnum={arg_position}).'
            )
        return self.back_funcs[key]

def multiply_back0(grad_out, out, x, y):
    if not isinstance(y, t.Tensor):
        y = t.tensor(y)
    return unbroadcast(grad_out * y, x)

def multiply_back1(grad_out, out, x, y):
    if not isinstance(x, t.Tensor):
        x = t.tensor(x)
    return unbroadcast(grad_out * x, y)

def cx16_build_lookup():
    bf = BackwardFuncLookup()
    bf.add_back_func(t.multiply, 0, multiply_back0)
    bf.add_back_func(t.multiply, 1, multiply_back1)
    return bf
```

Symmetric ops still register twice — mirror bodies, separate (fn, argnum) keys. The dispatcher is the audience: it looks up by argnum regardless of the op's mathematical symmetry.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["Backprop: BackwardFuncLookup", "Backprop: multiply_back", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()